<a href="https://colab.research.google.com/github/dmainagithub/LLMs-Lessons/blob/main/huggingface_text_classification_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification Tutorial

Note: a GPU is needed in google colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU

## 2. Import necessary commands

In [1]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for upgrade
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")



Using transformers version: 5.16.1
Using datasets version: 5.0.1
Using torch version: 2.11.0+cu128


## 3. Getting a dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [3]:
# What features are there
dataset.column_names

{'train': ['text', 'label']}

In [4]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [5]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [6]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f" Text: {text} | Label: {label}")


[199, 66, 85, 192, 212]
[INFO] Random samples from dataset:

 Text: Crunchy sushi roll with a creamy filling, featuring shrimp tempura and avocado. | Label: food
 Text: Cuddling with a cat on her lap, a woman enjoys her morning coffee | Label: not_food
 Text: Set of mixing bowls perched on a shelf | Label: not_food
 Text: Set of spoons stored in a drawer | Label: not_food
 Text: Black and white checkered kitchen floor adding a classic touch | Label: not_food


In [7]:
range(len(dataset["train"]))

range(0, 250)

In [8]:
dataset["train"].unique("label")

['food', 'not_food']

In [9]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])


Counter({'food': 125, 'not_food': 125})

In [10]:
# Turn our dataset into a dataframe
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
220,Sewing machine ready for use on a table,not_food
211,A boy giving his dog a bath in the backyard,not_food
195,Bicycle leaning casually against a wall,not_food
49,"Pizza with a white sauce base, topped with spi...",food
92,Yoga mat rolled up and ready in a corner,not_food
45,Smoky flavored sushi roll with smoked salmon o...,food
39,A bowl of sliced cucumbers with a sprinkle of ...,food


In [11]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

1. Tokenization (machines prefer numbers rather than words)

2. Creating a train-test split (train split for training and test split for evaluation)

In [12]:
# Create a mapping programmatically
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [13]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
  print(idx, label)
  id2label[idx] = label

0 not_food
1 food


In [14]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample ={"text": "This is a sentence about my favorite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favorite food: honey', 'label': 1}

In [15]:
# Map our dataset labels to numbers (the whole dataset)
# With dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [16]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

{'text': ['Stack of books waiting to be read on a bookshelf',
  'Microwave oven ready for use on a kitchen counter',
  'A colorful bowl of mixed carrots, including orange and purple.',
  'A close-up shot of a big orange pumpkin with a face cut out of the side for Halloween.',
  'Tangy fish curry bowl, featuring delicate fish pieces in a zesty sauce made with tamarind and curry leaves, ideal for a light meal.'],
 'label': [0, 0, 1, 1, 1]}

### Train Test Split

* https://huggingface.co/docs/datasets/v4.8.4/process  

In [17]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [18]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Celery in a bowl, served with a side of peanut butter and a sprinkle of raisins for a classic, tasty snack.',
 'label': 1}

In [19]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["train"][random_idx_test]
random_sample_test

{'text': 'Pizza with a Mediterranean twist, featuring toppings like feta cheese, kalamata olives, and roasted red peppers',
 'label': 1}

## Tokenization

https://platform.openai.com/tokenizer - open ai tokenizers.
https://github.com/huggingface/tokenizers - huggingface tokenizers.
To do this locally you need to have rust installed.
* RUST is a programming language: https://rust-lang.org/
* Huggingface auto classes: https://huggingface.co/docs/transformers/en/model_doc/auto

To find all the models: https://huggingface.co/models

We will use this specific one: https://huggingface.co/distilbert/distilbert-base-uncased

** Models are often paired with tokenizers.
* Tokenizers = turn text to numbers.
* Models = find patterns in those numbers.


In [57]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [21]:
# Test our tokenizer
# Open ai token ids for "I love pizza" = [[40, 3047, 27941]]
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

### Tokenizer vocabulary and input sequence

In [22]:
# Tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer_vocab}")

# Maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

[INFO] Number of items in our tokenizer vocab: 30522
[INFO] Max tokenizer input sequence length: 512


In [24]:
# Does Nderitu occur in the vocab?
# tokenizer.vocab["nderitu"]

In [ ]:
# tokenizer.vocab

In [25]:
tokenizer("Nderitu")

{'input_ids': [101, 1050, 4063, 4183, 2226, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [26]:
tokenizer.convert_ids_to_tokens(tokenizer("Nderitu").input_ids)

['[CLS]', 'n', '##der', '##it', '##u', '[SEP]']

In [27]:
# Tokenizing an emoji
tokenizer.convert_ids_to_tokens(tokenizer("👍").input_ids)

['[CLS]', '[UNK]', '[SEP]']

In [29]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [30]:
# Get 5 random items from the vocab
import random

random.sample(sorted(tokenizer.vocab.items()), k=7)

[('##sha', 7377),
 ('emi', 12495),
 ('akron', 22735),
 ('##kko', 22426),
 ('routing', 16972),
 ('realistic', 12689),
 ('[unused768]', 773)]

## Part 5: Preparing text data for use with a model

* Preprocessing function

In [31]:
def tokenize_text(example):
  """
  Tokenize given example text and return the tokenized text.
  """
  return tokenizer(example["text"],
                   padding=True,
                   truncation=True)

In [32]:
tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [33]:
example_sample_2 = {"text": "I love my wife", "label": 0}

# Test the function
tokenize_text(example_sample_2)

{'input_ids': [101, 1045, 2293, 2026, 2564, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [34]:
long_text = "I love you very much " * 1000
len(long_text)

21000

In [35]:
tokenize_long_text = tokenize_text({"text": long_text, "label": 0})
len(tokenize_long_text["input_ids"])

512

### Tokenize our dataset

In [36]:
# Map our tokenize text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)
tokenized_dataset

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50
    })
})

In [37]:
tokenizer.all_special_tokens

['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']

In [38]:
tokenizer.all_special_ids

[100, 102, 0, 101, 103]

In [39]:
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
  print(f"[INFO] Key: {key}")
  print(f"Train sample: {train_tokenized_sample[key]}")
  print(f"Test sample: {test_tokenized_sample[key]}")
  print()

[INFO] Key: text
Train sample: Set of headphones placed on a desk
Test sample: A slice of pepperoni pizza with a layer of melted cheese

[INFO] Key: label
Train sample: 0
Test sample: 1

[INFO] Key: input_ids
Train sample: [101, 2275, 1997, 2132, 19093, 2872, 2006, 1037, 4624, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [101, 1037, 14704, 1997, 11565, 10698, 10733, 2007, 1037, 6741, 1997, 12501, 8808, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[INFO] Key: token_type_ids
Train sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[INFO] Key: attention_mask
Train sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Test sample: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0

### Compute accuracy: evaluation metrics

In [54]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Computes the accuracy of a model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels
  return accuracy_metric.compute(predictions=predictions, references=labels)

In [42]:
example_preds_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_preds_one_incorrect = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_preds_all_correct, example_labels))}")
print(f"Accuracy when one prediction is incorrect: {compute_accuracy((example_preds_one_incorrect, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when one prediction is incorrect: {'accuracy': 0.9}


## Model training workflow

### Model

In [44]:
id2label

{0: 'not_food', 1: 'food'}

In [45]:
label2id

{'not_food': 0, 'food': 1}

In [43]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### Counting the parameters of our model

In [46]:
def count_params(model):
  """
  Count the parameters of a PyTorch model
  """
  trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad)
  total_parameters = sum(param.numel() for param in model.parameters())

  return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

count_params(model)

{'trainable_parameters': 66955010, 'total_parameters': 66955010}

In [49]:
# list(model.parameters())

### Part 3: Creating a folder to save our model

In [50]:
from pathlib import Path

# Create models dir
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "hf_food_not_food_text_classifier-distilbert-base-uncased"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir

PosixPath('models/hf_food_not_food_text_classifier-distilbert-base-uncased')

### Training arguments

In [51]:
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints: {model_save_dir}")

BATCH_SIZE = 32

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    # hub_token="my_HF_token",
    # push_to_hub=True,
    hub_private_repo=False
)

[INFO] Saving model checkpoints: models/hf_food_not_food_text_classifier-distilbert-base-uncased


In [55]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Computes the accuracy of a model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

### Setting up our trainer

In [60]:
from transformers import Trainer

# Setting up the trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset= tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_accuracy
)

trainer

### Train the model

In [62]:
results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000506,0.000372,1.000000
2,0.000417,0.000312,1.000000
3,0.000361,0.000271,1.000000
4,0.000303,0.000243,1.000000
5,0.000299,0.000222,1.000000
6,0.000261,0.000208,1.000000
7,0.000266,0.000197,1.000000
8,0.000251,0.000190,1.000000
9,0.000245,0.000186,1.000000
10,0.000239,0.000184,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [63]:
for key, value in results.metrics.items():
  print(f"{key} : {value}")

train_runtime : 101.0881
train_samples_per_second : 19.785
train_steps_per_second : 0.692
total_flos : 18110777160000.0
train_loss : 0.00031464336373444114
epoch : 10.0


### Saving the model for later use

In [64]:
# Save the model
print(f"[INFO] Saving the model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

[INFO] Saving the model to models/hf_food_not_food_text_classifier-distilbert-base-uncased


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]